# Transfer-time scoring on Kaggle CPU

Second task of EarlyMule-TG: every outgoing transfer is scored just before it executes and the riskiest are held (`python -m earlymule.models.transfer_lgbm train`, variants in `configs/transfer_scoring.yaml`). CPU only, so it does not use the weekly GPU quota. The (variant, seed) pairs in `JOBS` run one after another; each writes scores, per-mule hold tables, `model.txt`, `importance.csv`, `metrics.json` and `config.json` to `/kaggle/working/transfer_lgbm_<variant>/seed<N>/` as soon as it ends. The runs are logged locally with `python -m earlymule.models.transfer_lgbm import --from <downloaded folder>`.

Input: the private dataset `verzol/earlymule-80k-transfer-task` (made by `python -m earlymule.models.transfer_lgbm export`). One notebook per batch is generated by `notebooks/kaggle/transfer_scoring/make_batches.py`, which rewrites the parameter cell.

In [ ]:
# Run parameters: (variant, seed) pairs, run one after another; commit to check out (None = latest main)
JOBS = [('sender', 0)]
COMMIT = None

In [ ]:
# Clone the repository at COMMIT and install it without dependencies (LightGBM, pandas and PyYAML are preinstalled)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!python -c "import lightgbm, pandas; print('lightgbm', lightgbm.__version__, 'pandas', pandas.__version__)"
!nproc; free -g

The dataset folder is found by its manifest, wherever Kaggle mounts it. Each run is its own process, so its memory is released before the next one starts; logs go next to the outputs.

In [ ]:
# Train every (variant, seed) pair in order; a failed run is logged and the next one still starts
import json, subprocess, time
from pathlib import Path
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if (m.parent / 'first_outflows.parquet').exists())
codes = []
for variant, seed in JOBS:
    out = Path('/kaggle/working') / f'transfer_lgbm_{variant}' / f'seed{seed}'
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    cmd = ['python', '-m', 'earlymule.models.transfer_lgbm', 'train', '--data', str(data), '--variant', variant,
           '--seed', str(seed), '--out', str(out)]
    code = subprocess.call(cmd, cwd='/tmp/earlymule-tg', stdout=open(out / 'run.log', 'w'), stderr=subprocess.STDOUT)
    codes.append((variant, seed, code, round((time.time() - t0) / 60, 1)))
    print(codes[-1], flush=True)
print(codes)

In [ ]:
# Last line of every run log (the run summary)
for log in sorted(Path('/kaggle/working').glob('transfer_lgbm_*/seed*/run.log')):
    print(f'--- {log.parent.parent.name}/{log.parent.name}')
    print(log.read_text()[-800:])